# 01 · Data Wrangling with Pandas and Polars

> 🇪🇸 Versión en español: [`01_pandas_y_polars.ipynb`](01_pandas_y_polars.ipynb)

Before training any model you need to **load, explore, clean and transform** the data.
Here every operation is done twice: once with **Pandas** (the long-standing standard) and once with **Polars**
(newer, written in Rust, multi-threaded and with *lazy* evaluation).

**Goals**
1. Load a CSV and inspect it.
2. Detect and handle missing values.
3. Filter, create columns and group.
4. Understand the Polars expression API and *lazy* mode.
5. Convert between both libraries.

> Dataset: `datos/houses.csv` (synthetic, generated by `datos/generar_datos.py`). Target: `price`.

In [1]:
# Libraries
#=====================================================
import time

import numpy as np
import pandas as pd
import polars as pl

print("pandas", pd.__version__, "| polars", pl.__version__)
PATH = "datos/houses.csv"

pandas 3.0.6 | polars 1.44.2


## 1. Load and inspect

In [2]:
# Pandas
#=====================================================
df_pd = pd.read_csv(PATH)
df_pd.head()

,id,neighborhood,sqm,rooms,age,distance_center_km,has_garage,price
0,1,South,74.1,1,22.0,24.31,yes,41500.0
1,2,North,141.8,3,8.0,9.23,no,285500.0
2,3,Outskirts,67.9,3,32.0,13.96,yes,49700.0
3,4,South,100.0,2,33.0,13.05,no,77900.0
4,5,Downtown,89.5,2,44.0,0.19,no,133400.0


In [3]:
# Polars
#=====================================================
df_pl = pl.read_csv(PATH)
df_pl.head()

id,neighborhood,sqm,rooms,age,distance_center_km,has_garage,price
i64,str,f64,i64,f64,f64,str,f64
1,"""South""",74.1,1,22.0,24.31,"""yes""",41500.0
2,"""North""",141.8,3,8.0,9.23,"""no""",285500.0
3,"""Outskirts""",67.9,3,32.0,13.96,"""yes""",49700.0
4,"""South""",100.0,2,33.0,13.05,"""no""",77900.0
5,"""Downtown""",89.5,2,44.0,0.19,"""no""",133400.0


In [4]:
# Data types and summary statistics
#=====================================================
df_pd.info()
df_pd.describe()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  2000 non-null   int64  
 1   neighborhood        2000 non-null   str    
 2   sqm                 1920 non-null   float64
 3   rooms               2000 non-null   int64  
 4   age                 1940 non-null   float64
 5   distance_center_km  2000 non-null   float64
 6   has_garage          2000 non-null   str    
 7   price               2000 non-null   float64
dtypes: float64(4), int64(2), str(2)
memory usage: 142.5 KB


,id,sqm,rooms,age,distance_center_km,price
count,2000.000000,1920.000000,2000.000000,1940.000000,2000.000000,2000.000000
mean,1000.500000,111.080156,3.194500,29.060825,11.145635,159529.050000
std,577.494589,34.613266,1.283937,17.236558,7.679769,104147.157841
min,1.000000,35.000000,1.000000,0.000000,0.020000,30000.000000
25%,500.750000,87.150000,2.000000,15.000000,3.287500,75300.000000
50%,1000.500000,110.650000,3.000000,29.000000,10.645000,143900.000000
75%,1500.250000,134.250000,4.000000,44.000000,17.842500,224900.000000
max,2000.000000,218.100000,7.000000,59.000000,24.990000,574900.000000


In [5]:
print(df_pl.schema)
df_pl.describe()

Schema([('id', Int64), ('neighborhood', String), ('sqm', Float64), ('rooms', Int64), ('age', Float64), ('distance_center_km', Float64), ('has_garage', String), ('price', Float64)])


statistic,id,neighborhood,sqm,rooms,age,distance_center_km,has_garage,price
str,f64,str,f64,f64,f64,f64,str,f64
"""count""",2000.0,"""2000""",1920.0,2000.0,1940.0,2000.0,"""2000""",2000.0
"""null_count""",0.0,"""0""",80.0,0.0,60.0,0.0,"""0""",0.0
"""mean""",1000.5,null,111.080156,3.1945,29.060825,11.145635,null,159529.05
"""std""",577.494589,null,34.613266,1.283937,17.236558,7.679769,null,104147.157841
"""min""",1.0,"""Downtown""",35.0,1.0,0.0,0.02,"""no""",30000.0
"""25%""",501.0,null,87.2,2.0,15.0,3.29,null,75300.0
"""50%""",1001.0,null,110.7,3.0,29.0,10.65,null,144000.0
"""75%""",1500.0,null,134.2,4.0,44.0,17.84,null,224800.0
"""max""",2000.0,"""South""",218.1,7.0,59.0,24.99,"""yes""",574900.0


## 2. Missing values

Always check how many values are missing **per column** before modeling.
Pandas uses `NaN`; Polars distinguishes between `null` (missing) and `NaN` (invalid numeric result).

In [6]:
print("Pandas:\n", df_pd.isna().sum(), sep="")
print("\nPolars:")
df_pl.null_count()

Pandas:
id                     0
neighborhood           0
sqm                   80
rooms                  0
age                   60
distance_center_km     0
has_garage             0
price                  0
dtype: int64

Polars:


id,neighborhood,sqm,rooms,age,distance_center_km,has_garage,price
u32,u32,u32,u32,u32,u32,u32,u32
0,0,80,0,60,0,0,0


In [7]:
# Impute with the median (note: in real ML the median is computed ONLY on train, see notebook 02)
#=====================================================
df_pd_clean = df_pd.fillna({
    "sqm": df_pd["sqm"].median(),
    "age": df_pd["age"].median(),
})

df_pl_clean = df_pl.with_columns(
    pl.col("sqm").fill_null(pl.col("sqm").median()),
    pl.col("age").fill_null(pl.col("age").median()),
)
print(df_pd_clean.isna().sum().sum(), df_pl_clean.null_count().sum_horizontal().item())

0 0


## 3. Filter, create columns and sort

In Polars everything is written as **expressions** (`pl.col(...)`) inside contexts:
`select`, `with_columns`, `filter`, `group_by(...).agg(...)`.

In [8]:
# Pandas: Downtown houses larger than 100 m², with price per m²
#=====================================================
res_pd = (
    df_pd_clean
    .assign(price_sqm=lambda d: d["price"] / d["sqm"])
    .query("neighborhood == 'Downtown' and sqm > 100")
    .sort_values("price_sqm", ascending=False)
    [["neighborhood", "sqm", "price", "price_sqm"]]
)
res_pd.head()

,neighborhood,sqm,price,price_sqm
340,Downtown,110.65,387100.0,3498.418437
1296,Downtown,108.40,368400.0,3398.523985
85,Downtown,142.30,476100.0,3345.748419
434,Downtown,108.40,362300.0,3342.250923
1340,Downtown,131.70,439600.0,3337.889142


In [9]:
# Polars: the same thing
#=====================================================
res_pl = (
    df_pl_clean
    .with_columns((pl.col("price") / pl.col("sqm")).alias("price_sqm"))
    .filter((pl.col("neighborhood") == "Downtown") & (pl.col("sqm") > 100))
    .sort("price_sqm", descending=True)
    .select("neighborhood", "sqm", "price", "price_sqm")
)
res_pl.head()

neighborhood,sqm,price,price_sqm
str,f64,f64,f64
"""Downtown""",110.65,387100.0,3498.418437
"""Downtown""",108.4,368400.0,3398.523985
"""Downtown""",142.3,476100.0,3345.748419
"""Downtown""",108.4,362300.0,3342.250923
"""Downtown""",131.7,439600.0,3337.889142


## 4. Group and aggregate

In [10]:
# Pandas
#=====================================================
df_pd_clean.groupby("neighborhood").agg(
    n=("price", "size"),
    mean_price=("price", "mean"),
    mean_sqm=("sqm", "mean"),
).sort_values("mean_price", ascending=False)

,n,mean_price,mean_sqm
neighborhood,,,
Downtown,408,262992.156863,111.455025
North,600,159701.000000,110.656500
South,579,125191.364421,111.138515
Outskirts,413,105207.990315,111.160169


In [11]:
# Polars
#=====================================================
df_pl_clean.group_by("neighborhood").agg(
    pl.len().alias("n"),
    pl.col("price").mean().alias("mean_price"),
    pl.col("sqm").mean().alias("mean_sqm"),
).sort("mean_price", descending=True)

neighborhood,n,mean_price,mean_sqm
str,u32,f64,f64
"""Downtown""",408,262992.156863,111.455025
"""North""",600,159701.0,110.6565
"""South""",579,125191.364421,111.138515
"""Outskirts""",413,105207.990315,111.160169


## 5. Polars *lazy* mode

With `scan_csv` Polars reads nothing until you call `.collect()`. Meanwhile it builds a
**query plan** and optimizes it (e.g. it only reads the columns you use and pushes filters down early).
This is key for large files.

In [12]:
query = (
    pl.scan_csv(PATH)
    .filter(pl.col("has_garage") == "yes")
    .group_by("neighborhood")
    .agg(pl.col("price").median().alias("median_price"))
    .sort("median_price")
)
print(query.explain())   # optimized plan
query.collect()

SORT BY [col("median_price")]
  AGGREGATE[maintain_order: false]
    [col("price").median().alias("median_price")] BY [col("neighborhood")]
    FROM
    simple π 2/2 ["neighborhood", "price"]
      Csv SCAN [datos/houses.csv]
      PROJECT 3/8 COLUMNS
      SELECTION: col("has_garage") == "yes"
      ESTIMATED ROWS: 2191


neighborhood,median_price
str,f64
"""Outskirts""",109000.0
"""South""",127200.0
"""North""",168550.0
"""Downtown""",269000.0


## 6. Quick speed comparison

With 2,000 rows the difference is negligible, so we replicate the data to notice it.
Timings depend on your machine.

In [13]:
big_pd = pd.concat([df_pd_clean] * 500, ignore_index=True)
big_pl = pl.from_pandas(big_pd)
print(f"{len(big_pd):,} rows")

t = time.perf_counter()
big_pd.groupby("neighborhood")["price"].agg(["mean", "std", "median"])
print(f"pandas: {time.perf_counter() - t:.3f} s")

t = time.perf_counter()
big_pl.group_by("neighborhood").agg(
    pl.col("price").mean(), pl.col("price").std().alias("std"), pl.col("price").median().alias("median")
)
print(f"polars: {time.perf_counter() - t:.3f} s")

1,000,000 rows


pandas: 0.249 s
polars: 0.085 s


## 7. Converting between Pandas, Polars and NumPy

Scikit-learn accepts Pandas and Polars directly; PyTorch needs NumPy arrays (or tensors).

In [14]:
polars_to_pandas = df_pl_clean.to_pandas()
pandas_to_polars = pl.from_pandas(df_pd_clean)
matrix = df_pl_clean.select("sqm", "rooms").to_numpy()
print(type(polars_to_pandas), type(pandas_to_polars), matrix.shape, matrix.dtype)

<class 'pandas.DataFrame'> <class 'polars.dataframe.frame.DataFrame'> (2000, 2) float64


## Pandas ↔ Polars cheat sheet

| Task | Pandas | Polars |
|---|---|---|
| Read CSV | `pd.read_csv(p)` | `pl.read_csv(p)` / `pl.scan_csv(p)` |
| Select | `df[["a", "b"]]` | `df.select("a", "b")` |
| Filter | `df[df.a > 0]` / `df.query("a > 0")` | `df.filter(pl.col("a") > 0)` |
| New column | `df.assign(c=df.a * 2)` | `df.with_columns((pl.col("a") * 2).alias("c"))` |
| Group | `df.groupby("k").agg(m=("a", "mean"))` | `df.group_by("k").agg(pl.col("a").mean())` |
| Missing values | `df.isna().sum()` / `fillna` | `df.null_count()` / `fill_null` |
| Sort | `df.sort_values("a")` | `df.sort("a")` |

## ✏️ Exercises
1. Using both libraries, compute the mean price by number of `rooms` and by `neighborhood` at the same time.
2. Create an `age_category` column (`"new"` < 10 years, `"mid"` < 30, `"old"` otherwise).
   Polars hint: `pl.when(...).then(...).otherwise(...)`.
3. Using `scan_csv`, get the 5 most expensive houses in each neighborhood (hint: `group_by(...).head(5)` after sorting).